In [0]:
%sql
--customers
CREATE OR REPLACE TABLE ecommerce.silver.customers AS
SELECT
  bcustomers.customer_id,
  bcustomers.name,
  lower(trim(bcustomers.email)) AS email,
  bcustomers.phone,
  CASE WHEN bcustomers.state IN ('SP','RJ','MG','RS','PR','SC','BA','PE','CE','DF')
       THEN bcustomers.state END AS state,
  CAST(bcustomers.created_at AS TIMESTAMP) AS created_at,
  make_date(bcustomers.year, bcustomers.month, bcustomers.day) AS data_carga
FROM ecommerce.bronze.customers bcustomers
WHERE bcustomers.customer_id IS NOT NULL
QUALIFY ROW_NUMBER() OVER (
  PARTITION BY bcustomers.customer_id
  ORDER BY CAST(bcustomers.created_at AS TIMESTAMP) DESC
) = 1;

In [0]:
%sql
-- products
CREATE OR REPLACE TABLE ecommerce.silver.products AS
SELECT
  bproducts.product_id,
  bproducts.product_name,
  COALESCE(bproducts.category, 'Não informado') AS category,
  CASE WHEN bproducts.price > 0 THEN bproducts.price END AS price,
  bproducts.is_active,
  make_date(bproducts.year, bproducts.month, bproducts.day) AS data_carga
FROM ecommerce.bronze.products bproducts
WHERE bproducts.product_id IS NOT NULL
QUALIFY ROW_NUMBER() OVER (
  PARTITION BY bproducts.product_id
  ORDER BY make_date(bproducts.year, bproducts.month, bproducts.day) DESC
) = 1;

In [0]:
%sql
--orders
CREATE OR REPLACE TABLE ecommerce.silver.orders AS
SELECT
  borders.order_id,
  borders.customer_id,
  CAST(borders.order_date AS TIMESTAMP) AS order_date,
  CAST(borders.order_date AS DATE)      AS order_day,
  CASE WHEN borders.status IN ('completed','pending','canceled')
       THEN borders.status ELSE 'unknown' END AS status,
  borders.payment.method       AS payment_method,
  borders.payment.installments AS installments,
  size(borders.items)          AS qtd_itens,
  make_date(borders.year, borders.month, borders.day) AS data_carga
FROM ecommerce.bronze.orders borders
WHERE borders.order_id IS NOT NULL
QUALIFY ROW_NUMBER() OVER (
  PARTITION BY borders.order_id
  ORDER BY CAST(borders.order_date AS TIMESTAMP) DESC
) = 1;

In [0]:
%sql
--items
CREATE OR REPLACE TABLE ecommerce.silver.order_items AS
SELECT
  o.order_id,
  item.product_id,
  item.quantity,
  item.unit_price,
  item.quantity * item.unit_price AS total_item,
  o.data_carga
FROM (
  SELECT
    borders.order_id,
    borders.items,
    make_date(borders.year, borders.month, borders.day) AS data_carga
  FROM ecommerce.bronze.orders borders
  WHERE borders.order_id IS NOT NULL
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY borders.order_id
    ORDER BY CAST(borders.order_date AS TIMESTAMP) DESC
  ) = 1
) o
LATERAL VIEW explode(o.items) i AS item
WHERE item.quantity > 0;

In [0]:
%sql
CREATE OR REPLACE TABLE ecommerce.silver.events AS
SELECT
  bevents.event_id,
  bevents.session_id,
  bevents.customer_id,
  bevents.event_type,
  bevents.product_id,
  bevents.platform,
  CAST(bevents.event_timestamp AS TIMESTAMP) AS event_timestamp,
  make_date(bevents.year, bevents.month, bevents.day) AS data_carga
FROM ecommerce.bronze.events bevents
WHERE bevents.event_id IS NOT NULL
QUALIFY ROW_NUMBER() OVER (
  PARTITION BY bevents.event_id
  ORDER BY CAST(bevents.event_timestamp AS TIMESTAMP) DESC
) = 1;